# Finger Gait Idea Plan

## 1. 当前阶段性结论

经过对 `Siekmann 2021`、`Khandate 2022`、`Bouyarmane 2011` 的阅读和多轮讨论，当前最稳的判断是：

- 不要把 hand 内 finger gait 硬类比成双足的 `walking/running/hopping/skipping` 那种丰富 gait family。
- 对当前手内旋转任务，更合理的表述是：
  - finger gait 更像一个 **low-dimensional, strongly constrained contact scheduling manifold**；
  - 重要的不是 gait 名称数量，而是 **contact handoff 是否能被显式参数化、条件化和调度**。
- 因此，研究重点应从“是否会自然涌现 finger gait”转向“是否能把 finger gait 从 emergent behavior 变成 commanded schedule”。

一句话总结：

> 我们更像是在研究 `phase-conditioned contact scheduling`，而不是在研究“手里有多少种 common gaits”。

## 2. 和 Khandate 2022 的关键区别

`Khandate 2022` 已经证明：

- 只用 intrinsic sensing（proprioception + tactile）可以学出 finger-gaiting / finger-pivoting；
- 连续绕轴重定向是可行的；
- exploration / initial state distribution 非常关键。

因此，如果当前工作只是：

- `RL + tactile/proprioception`，或
- `continuous in-hand rotation`，或
- `finger-gaiting can emerge`

那么区别性不够。

真正可成立的区别应当是：

### 方法论区别

1. 将 gait 从 **隐式涌现行为** 提升为 **显式条件变量**。
2. 将 reward 从 task-level 旋转奖励扩展为 **per-finger, phase-aware reward composition**。
3. 研究对象从 task success 转向 **contact schedule fidelity / schedule manifold**。

### 科学问题区别

1. 稳定主循环顺序是否存在，并且是否能被显式调参？
2. 3 指参与 vs 4 指参与，是否对应不同 schedule mode？
3. 内部 finger gait cadence 和外部 object rotation speed 是否可解耦？
4. 在 palm-supported 场景下，finger gait 是否更自然地写成 stance/contact transition sequence？

### 实验设定区别

1. 当前任务更接近 `palm-supported in-hand rotation`，而不是 purely unsupported precision grasp。
2. 可研究的变量不再只是“能否转动”，而是：
   - 参与手指集合
   - 交互顺序
   - duty factor / phase width
   - phase offset
   - cadence

## 3. 当前最稳妥的 claim

当前不建议做的 claim：

- hand 里也有像双足一样丰富的 gait taxonomy；
- 我们首次让 RL 学会 finger gaiting；
- 仅凭 palm-supported 设定变化就形成新意。

当前建议主打的 claim：

> 在 LeapHand 的手内旋转任务中，finger gait 可以被表述为一个显式的、可条件化的 contact scheduling 问题；策略不只是学会“转”，而是学会跟随由参与手指集合、交互顺序和相位参数定义的接触时序结构。

更聚焦一点的说法：

> Khandate 2022 证明了 finger gait 可以涌现；我们希望证明 finger gait 可以被命令。

## 4. 当前最小可行研究问题（MVP）

建议的 MVP 问题是：

> 在 LeapHand 的 palm-supported 单轴 in-hand rotation 任务中，是否存在一个可重复的主接触循环，使得通过显式条件化的 schedule 参数，可以稳定调节内部 finger contact-handoff，而不只是间接改变物体旋转速度，并且相较于无结构 RL 具有更好的可控性？

这里的 schedule 参数可先写成：

- 参与手指集合：$S$
- 交互顺序：$\pi$
- 每根手指的 duty factor / support ratio：$r_i$
- 每根手指的 phase offset：$\phi_i$
- 可选的内部 cadence：$\omega_g$

当前建议先把 `cadence` 放到第二优先级，第一阶段先聚焦：

- $S$
- $\pi$
- $r_i$
- $\phi_i$

原因：这些变量先决定“主循环结构”，而 cadence 更像是在既定 schedule 上调节节律。

## 5. 计划阶段应优先明确的变量

在正式制定实验计划前，最先需要钉住的是以下 4 类变量：

### A. 参与手指集合

先考虑：

- 4 指参与
- 3 指参与

核心问题：

- 不同对象 / 任务下是否存在偏好的 active finger subset？
- 3 指 vs 4 指是否对应稳定性和灵活性的不同权衡？

### B. 交互顺序

例如：

- 固定顺序
- 逆序
- 是否允许少数顺序模式切换

核心问题：

- 稳定主循环顺序是否基本唯一？
- 顺序能否被命令并被策略实际执行？

### C. 相位参数

先考虑：

- duty factor / phase width：$r_i$
- phase offset：$\phi_i$

核心问题：

- 改变这些参数后，per-finger contact histogram 是否随之移动？
- schedule agreement 是否上升？

### D. 内部节律与外部输出

后续再重点研究：

- 内部 cadence：$\omega_g$
- 外部 object rotation speed：$\omega_o$

核心问题：

- $\omega_g$ 与 $\omega_o$ 是否不是一一对应？
- 是否存在“内部 finger gait 更快，但外部旋转并不等比例更快”的 regime？

## 6. 进入制定计划前的默认方向

如果下一轮开始正式做计划，当前默认方向是：

1. 先限制到 `single-axis`, `palm-supported`, `small object family`。
2. 先不追求丰富 gait catalogue，而先证明：
   - 稳定主循环顺序存在；
   - 它可被 `S, \pi, r_i, \phi_i` 显式调参。
3. 实验至少要有一个 `unstructured RL` baseline，证明 schedule conditioning 的意义不只是换了一种写法。
4. 关键评估不只看 task return，还要看：
   - per-finger contact phase histogram
   - schedule agreement
   - unscheduled contact ratio
   - regrasp success / timing
   - drop rate
   - command switch controllability

## 7. 当前阶段的一句话决定

当前阶段已经可以进入“制定计划”模式，但计划不应再围绕“还有哪些 gait 名字”展开，而应围绕：

> 如何把 `S, \pi, r_i, \phi_i` 变成一个可验证、可调参、可比较 baseline 的 schedule-conditioned manipulation framework。

## 8. 最新反馈补充

用户新增了两个非常重要的方向约束：

### A. 最终目标仍然是 single policy for common gaits

这意味着：

- $S, \pi, r_i, \phi_i$ 不应被理解为只能逐个训练多个 policy；
- 更合理的目标是：**一个统一策略** 接收这些 schedule 参数作为条件输入；
- 前面提到的“先固定某个变量再扫另一个变量”，应该只被视为 **实验分析与识别贡献的手段**，而不是最终方法形态。

因此，计划阶段的更准确表述应为：

> 最终方法是单策略、多命令；实验阶段可以分块扫参，用来判断每个 schedule 参数是否真的可控、可识别、可解释。

### B. 保留图相关元素作为潜在设计方向

用户希望保留图相关方法的可能性，例如：

- graph transformer
- 图注意力机制（GAT）

这个想法是合理的，因为 hand-object interaction 天然具有图结构：

- 节点：手指、手掌、物体、可选的接触状态节点
- 边：接触、相对位姿、载荷传递、近邻、是否处于 schedule 邻接关系

但当前阶段的建议是：

- 图模块可以保留为候选编码器或后续扩展；
- 不应在现阶段盖过 `phase-conditioned contact scheduling` 这条主线；
- 否则贡献会从“schedule 可以被命令”漂移成“又一篇图模型 manipulation paper”。

一个比较稳的思路是：

> 当前 paper 的 headline 仍然是 schedule-conditioned manipulation；图模块若加入，更适合作为 interaction encoder，而不是 headline novelty。

### C. 当前对计划结构的修正

因此，后续正式计划更适合写成：

1. 目标方法：`single policy for common gaits`
2. 条件变量：$S, \pi, r_i, \phi_i$，可选 `cadence`
3. 实验策略：分块扫参，而不是分裂成多个互不相关的小 policy
4. 图模块：暂作为候选编码器设计保留，待明确放在哪个环节

## 9. 图相关方向的补充判断（基于 T(R,O) Grasp）

阅读 `T(R,O) Grasp` 后，当前对图相关元素的判断如下：

### A. 它为什么和当前想法相关

`T(R,O) Grasp` 的核心不是 diffusion 本身，而是它强调了一点：

> robot-object interaction 可以被写成一个统一的图结构表示，而不是分别写成 robot-centric 或 object-centric 表示。

这对当前 finger gait 问题是有启发的，因为 hand-object contact scheduling 也天然是图结构：

- 节点：手指 link / fingertip、手掌、物体 patch
- 边：相对位姿、接触、接触力、载荷转移、近邻关系、schedule 邻接关系

因此，如果当前工作要加入图，最自然的用途不是“再做一篇 grasp diffusion”，而是：

> 用图来编码 hand-object interaction state，使 single policy 更容易表示 contact scheduling manifold。

### B. 为什么它不能直接成为当前工作的 headline

`T(R,O) Grasp` 主打的是：

- cross-embodiment grasping
- graph diffusion
- 高效 grasp synthesis

而当前 finger gait 工作主打的应该是：

- phase-conditioned contact scheduling
- schedule 可命令性
- 内部接触时序结构的可控与可解释

因此，如果图在当前 paper 中直接被提升为 headline，很容易把贡献重心带偏，变成：

- 一个图模型 manipulation paper，

而不是：

- 一个把 finger gait 显式化为 schedule 的 paper。

### C. 当前最合理的图落点

如果保留图相关元素，当前最合理的用法是：

1. **作为 policy encoder**
   - 输入图结构化的 hand-object interaction state
   - 输出一个 latent，用于 single policy 的 action head
   - 优点：符合“方法导向”，也和用户当前偏好一致

2. **但必须保留显式 schedule command**
   - 图只能帮助表示 interaction state
   - 不能替代 `S, \pi, r_i, \phi_i` 这些 schedule 变量本身
   - 否则会重新退回“让结构自己隐式涌现”的路线

3. **图更适合做 interaction prior，而不是 task definition**
   - 主问题仍然是 schedule-conditioned manipulation
   - 图只是帮助单策略更好地编码手-物体交互关系

### D. 当前可行的图表示草案

如果后续要把图写进计划，当前一个可行草案是：

- 节点：
  - fingertip / hand link nodes
  - palm node
  - object patch nodes
- 边特征：
  - relative SE(3)
  - contact binary / force
  - load estimate
  - schedule mask / phase activity
  - geometric adjacency
- 全局条件：
  - active finger subset `S`
  - order `\pi`
  - phase parameters `r_i, \phi_i`
  - optional cadence

这样图的角色是：

> 用结构化消息传递表示 interaction；而 schedule 变量继续作为显式命令输入。

### E. 当前阶段的结论

所以，图相关元素是值得保留的，而且它和“方法导向”并不冲突。

但当前最稳的边界仍然是：

> 图 = policy encoder / interaction prior
>
> schedule-conditioned contact scheduling = headline contribution

## 10. 多物体泛化与 RMA 的位置判断

用户新增的问题是：

> 当前工作最后是否应该测试不同物体？
>
> 是训练一个跨物体策略，还是分别为不同物体训练策略？
>
> `Qi 等 - 2022 - In-Hand Object Rotation via Rapid Motor Adaptation` 中的 object adaptation 思路是否值得借鉴？

当前判断如下：

### A. 不同物体测试是必须的，但不应自动升级为 headline novelty

如果当前 paper 完全只在单一物体上验证，说服力会明显不足。

因此：

- **多物体测试是需要的**；
- 但它更适合作为“schedule-conditioned 方法具有一定 object generalization”的验证；
- 不应一开始就把标题拉成“通用跨物体 hand manipulation foundation policy”。

更稳的做法是：

> 主贡献仍然是 schedule-conditioned single policy；
>
> object generalization 是一个重要但受控的验证维度。

### B. 不建议把“每个物体分别训练一个 policy”当主路线

如果每个物体都单独训练一个策略，会有两个问题：

1. 它削弱“common gaits / common schedule-conditioned policy”这个核心叙事；
2. 它更像是在证明“每个物体都有自己的最优策略”，而不是在证明“存在可共享的 schedule primitive”。

因此更合理的结构是：

- **主模型**：一个 single policy，训练在一组受控物体分布上；
- **per-object policy**：只作为 oracle / upper-bound / ablation baseline，而不是主方法。

### C. RMA 的启发在哪里

`Qi 等 - 2022` 给我们的真正启发，不只是“多物体泛化”，而是：

> 对当前任务真正重要的 object properties，可以被压缩成一个低维 latent，并由策略在线利用。

这和当前 finger gait 的问题是兼容的。

如果延伸到当前工作，可以把 object-related variation 分成两部分：

1. **几何 / 交互结构信息**
   - 哪些位置可能接触
   - 哪些 patch 更适合 handoff
   - 哪些 finger-object relation 几何上更自然
   - 这部分更适合由 **graph encoder** 表征

2. **隐含动力学信息**
   - 质量
   - 摩擦
   - 局部柔软度
   - COM 偏移
   - 这部分更适合由 **RMA-style adaptation latent** 从 proprio / tactile history 中估计

因此，图和 RMA 不是互斥的，它们更像是在处理不同类型的 object information。

### D. 当前 paper 最稳的 object generalization 方案

如果不想把问题做得过大，当前最稳的方案是：

1. 训练一个 **schedule-conditioned single policy**；
2. 训练分布覆盖一个 **小而受控的 object family**；
3. 测试时至少包含：
   - seen objects
   - held-out objects within family
   - 少量轻度 out-of-distribution objects
4. per-object policy 只作为对照或上界。

这样可以同时满足：

- 论文更有说服力；
- 不会让 object generalization 抢走主线；
- 仍然保留后续扩展到 RMA / graph adaptation 的空间。

### E. 对图和 RMA 的当前建议

当前阶段更推荐：

- **图作为 policy encoder**，负责 hand-object interaction representation；
- **RMA-style latent 作为后续可选模块**，负责 object hidden properties adaptation；
- 当前 paper 不一定要同时把两者都做重，否则主线会变得太散。

换句话说：

> 图更适合解决“如何表示 interaction”；
>
> RMA 更适合解决“如何适应未知 object properties”。

### F. 当前对计划的修正

因此，当前计划应默认：

1. 方法主线：schedule-conditioned single policy
2. 图：作为 interaction encoder 候选
3. 多物体：作为必要验证维度
4. per-object：仅作为 baseline
5. RMA-style adaptation：保留为后续增强项，或当多物体单策略性能不足时再引入

## 11. DexNDM 对图与 adaptation 放置位置的启发

阅读 `DexNDM` 前 10 页后，当前最有价值的结构启发不是它的完整 sim-to-real pipeline，而是它对“如何 factorize adaptation”的处理。

### A. DexNDM 的关键启发

`DexNDM` 的核心思想之一是：

> 不去直接学习整个 hand-object system 的整体高维 dynamics，而是将系统级影响压缩后，分解到 `joint-wise` 的动力学建模中。

这和当前我们讨论的图结构非常相关，因为它提示了一个重要判断：

- 如果 adaptation 信号是 `per-joint` 或 `per-link` 的，
- 那么它更自然地对应 **节点特征**，
- 而不是边特征。

### B. 对当前图设计的直接影响

因此，当前对图中不同信息的放置建议是：

#### 更适合节点特征的量

- per-joint / per-link adaptation latent
- finger proprioception summary
- local actuation / local hidden dynamics summary
- object-level latent（若是全局，可作为 object node 或 global token）

#### 更适合边特征的量

- relative SE(3)
- contact binary / contact force
- load transfer
- geometric adjacency
- schedule adjacency / handoff relation

换句话说：

> `adaptation latent` 更像 “这个节点当前处在什么局部动力学 regime”；
>
> `edge feature` 更像 “两个节点之间当前是什么关系”。

### C. 对当前 paper 的建议

如果后续真的引入 RMA-style 思想，当前更稳的实现是：

1. graph policy encoder 负责 interaction representation；
2. adaptation latent 若存在，优先注入到 fingertip / link node；
3. 通过 message passing 让局部 latent 影响全局 schedule-conditioned policy；
4. 不建议一开始把 adaptation latent 直接当主要 edge feature。

### D. 当前阶段的一句话判断

`DexNDM` 支持我们做出如下更清晰的结构选择：

> 当前 paper 如果保留图，图中“边”负责关系，“点”负责局部状态与 adaptation。
>
> 这和 `schedule-conditioned single policy` 的主线是相容的。

## 12. 当前已确认的计划边界

基于最新一轮反馈，当前已经明确的一点是：

- **多物体泛化 = 主验证，但非主贡献**。

这意味着：

1. 后续计划中应保留 `single policy` 跨一组受控物体的训练与测试；
2. 这会是核心实验之一；
3. 但论文 headline 仍应是 `schedule-conditioned single policy`，而不是“一个通用跨物体 manipulation foundation policy”。

当前阶段的主线边界可以简写为：

> 主贡献：phase-conditioned contact scheduling
>
> 关键验证：single policy 在多物体上的受控泛化
>
> 候选增强：graph encoder
>
> 暂缓模块：RMA-style adaptation